In [1]:
"""
Py-Microgrid Hybrid System Simulation Example
-----------------------------------
This example demonstrates how to:
1. Set up a hybrid system simulation
2. Download solar and wind resource data
3. Configure system parameters
4. Run optimization
5. Analyze and save results

Required files:
- Base YAML configuration file
- CSV file containing location data
"""

import os
import pandas as pd
from typing import Dict, List, Any

# Set NREL API key FIRST before any other imports to avoid timing issues
from py_microgrid.utilities.keys import set_developer_nrel_gov_key
set_developer_nrel_gov_key('ZaurwKOnwDUp8rMyNBIxI4XiBo3b7L5oruTi0VX3')

# Required imports (after API key is set)
from py_microgrid.utilities import ConfigManager
from py_microgrid.tools.optimization.system_optimizer import SystemOptimizer  
from py_microgrid.tools.optimization import LoadAnalyzer
# Corrected import path for the refactored EconomicCalculator
from py_microgrid.tools.optimization import EconomicCalculator
from py_microgrid.simulation.resource_files import ResourceDataManager

# Initialize resource manager for downloading data
resource_manager = ResourceDataManager(
    api_key='ZaurwKOnwDUp8rMyNBIxI4XiBo3b7L5oruTi0VX3',
    email='hanrong.h99@gmail.com'
)

# Location information
latitude = -33.5265
longitude = 149.1588

# Download resource data
solar_path = resource_manager.download_solar_data(
    latitude=latitude,
    longitude=longitude,
    year="2020" 
)
wind_path = resource_manager.download_wind_data(
    latitude=latitude,
    longitude=longitude,
    start_date="20200101",  
    end_date="20201231"
)

# Load and update YAML configuration
yaml_file_path = "./quick_start_config.yaml"  # Path to your main scenario config
config_manager = ConfigManager()
config = config_manager.load_yaml_safely(yaml_file_path)

# Update configuration with location and resource files
config['site']['data']['lat'] = latitude
config['site']['data']['lon'] = longitude
config['site']['solar_resource_file'] = solar_path.replace('\\', '/')  # Normalize path separators
config['site']['wind_resource_file'] = wind_path.replace('\\', '/')

# Save updated configuration
config_manager.save_yaml_safely(config, yaml_file_path)

# === INITIALIZE COMPONENTS USING PARAMETERS FROM YAML ===
# Read financial parameters directly from the YAML config file
financial_config = config.get('financial', {})
project_lifetime = financial_config.get('project_lifetime', 25)
discount_rate = financial_config.get('discount_rate', 5.88) / 100.0 # Convert from % to decimal

# Initialize the EconomicCalculator with values from the YAML file
economic_calculator = EconomicCalculator(
    discount_rate=discount_rate,
    project_lifetime=project_lifetime
)

# Initialize the SystemOptimizer
optimizer = SystemOptimizer(
    yaml_file_path=yaml_file_path,
    economic_calculator=economic_calculator,
    enable_flexible_load=True,
    max_load_reduction_percentage=0.2
)

# Define optimization bounds
config = config_manager.load_yaml_safely(yaml_file_path)
grid_enabled = config.get('technologies', {}).get('grid', {}).get('enabled', False)

if grid_enabled:
    # 6-component optimization: PV, Wind, Battery kWh, Battery kW, Genset, Grid
    bounds = [
        (5000, 50000),    # PV capacity (kW)
        (5, 50),          # Wind turbines
        (5000, 30000),    # Battery capacity (kWh)
        (1000, 10000),    # Battery power (kW)
        (5000, 20000),    # Genset capacity (kW)
        (5000, 20000)     # Grid capacity (kW)
    ]
else:
    # 5-component optimization: PV, Wind, Battery kWh, Battery kW, Genset
    bounds = [
        (5000, 50000),    # PV capacity (kW)
        (5, 50),          # Wind turbines
        (5000, 30000),    # Battery capacity (kWh)
        (1000, 10000),    # Battery power (kW)
        (5000, 20000)    # Genset capacity (kW)
    ]

# Define initial conditions (start at 10% of the range for each variable)
initial_conditions = [
    [bound[0] + (bound[1] - bound[0]) * 0.1 for bound in bounds]
]

print(f"Running optimization with {'6' if grid_enabled else '5'} components")
print(f"Grid enabled: {grid_enabled}")

# Run the optimization
result = optimizer.optimize_system(bounds, initial_conditions)

# Print final results
if result:
    print("\nOptimization Results:")
    print(f"PV Capacity: {result['PV Capacity (kW)']:.2f} kW")
    print(f"Wind Capacity: {result['Wind Turbine Capacity (kW)']:.2f} kW")
    print(f"Battery Capacity: {result['Battery Energy Capacity (kWh)']:.2f} kWh")
    print(f"Battery Power: {result['Battery Power Capacity (kW)']:.2f} kW")
    print(f"Genset Capacity: {result['Genset Capacity (kW)']:.2f} kW")
    if grid_enabled:
        print(f"Grid Capacity: {result['Grid Capacity (kW)']:.2f} kW")
    print(f"\nLCOE: ${result['System LCOE ($/kWh)']:.4f}/kWh")
    print(f"Net Present Cost: ${result['Net Present Cost ($)']:.2f}")
    print(f"CO2 Emissions: {result['Total CO2 emissions (tonne)']:.2f} tonnes")
    print(f"Demand Met: {result['Demand Met Percentage']:.2f}%")
else:
    print("Optimization failed to find a solution")
    
# Save optimization results to CSV
if result:
    results_df = pd.DataFrame([result])
    import os
    output_dir = os.getcwd()
    csv_filename = os.path.join(output_dir, f"optimization_results_{latitude}_{longitude}.csv")
    results_df.to_csv(csv_filename, index=False)
    print(f"✓ Optimization results saved to: {csv_filename}")
else:
    print("✗ No results to save - optimization failed")

print("✓ Quick start example completed successfully!")


/mnt/c/Users/Hanrong Huang/OneDrive - UNSW/Desktop/py_microgrid/py_microgrid/log/hybrid_systems_2025-07-18T12.03.27.198817.log
Using existing solar data file: /mnt/c/Users/Hanrong Huang/OneDrive - UNSW/Desktop/py_microgrid/py_microgrid/simulation/resource_files/solar/-33.5265_149.1588_psmv3_60_2020.csv
Using existing wind data file: /mnt/c/Users/Hanrong Huang/OneDrive - UNSW/Desktop/py_microgrid/py_microgrid/simulation/resource_files/wind/-33.5265_149.1588_NASA_2020_60min_50m.srw
✓ Successfully loaded cost configs from: /mnt/c/Users/Hanrong Huang/OneDrive - UNSW/Desktop/py_microgrid/py_microgrid/simulation/config
Running optimization with 5 components
Grid enabled: False

Calculating costs for: PV:9500kW, Wind:10000.0kW, Bat:7500kWh, Gen:6500kW
  PV Cost: $22,562,500
  Wind Cost: $35,000,000
  Battery Cost: $4,975,000
  Genset Cost: $436,566,923 (Fuel: $336,231,963, Replace: $50,700,000, Carbon: $7,335,461)

Calculating costs for: PV:9975kW, Wind:10000.0kW, Bat:7500kWh, Gen:6500kW
  PV